# 05 · Uplift Modelling — Ensemble 5 models (5-fold CV)

| # | Base model | Cấu hình |
|---|---|---|
| 1 | UPLIFT_RF_CIT · CFG028 | n=80, depth=8, leaf=120, treat=20, mf_cap=32 |
| 2 | UPLIFT_RF_CIT · CFG009 | n=120, depth=5, leaf=80, treat=30, mf_cap=32 |
| 3 | UPLIFT_RF_CIT · CFG011 | n=120, depth=5, leaf=160, treat=40, mf_cap=32 |
| 4 | INT_RF · RS_027 | n=560, depth=6, leaf=80, split=50, log2, entropy, balanced |
| 5 | INT_RF · RS_028 | n=420, depth=6, leaf=30, split=2, log2, gini, balanced |

`uplift_ensemble(x) = (uplift_1(x) + … + uplift_5(x)) / 5`, tính riêng cho cash và card.

Fold: `StratifiedKFold(5, shuffle=True, random_state=42)` stratify theo `treatment_cash × cash_sign_30d × card_sign_30d`.

In [ ]:
import sys
sys.path.append("..")

from src import config as C
from src.ensemble import cross_validate_ensemble
from src.plots import qini_plot
from src.preprocessing import load_model_data

X, W, y_cash, y_card, feat_cols = load_model_data()

In [ ]:
cv_summary, best = cross_validate_ensemble(X, W, y_cash, y_card)
cv_summary.round(6)

In [ ]:
cv_summary.to_csv(C.RESULTS_DIR / "ensemble_cv_qini.csv", index=False)

## Qini curve — fold tốt nhất theo `qini_sum`

In [ ]:
va = best["valid_idx"]
qini_plot(y_cash[va], (W[va] == 1).astype(int), best["uplift_cash"], "Cash Uplift Model — Qini Curve", C.FIGURES_DIR / "qini_curve_cash_ensemble_best_fold.png")
qini_plot(y_card[va], (W[va] == 0).astype(int), best["uplift_card"], "Card Uplift Model — Qini Curve", C.FIGURES_DIR / "qini_curve_card_ensemble_best_fold.png")